# Position shuffle test

A plain convolutional network on ciprofloxacin, with the variant columns in order and shuffled, on both collections.

**Before you start**

1. Settings, Accelerator: **GPU T4 x2**. Settings, Internet: **On**.
2. Add Input: the dataset **trilens-data**. The code is cloned from GitHub.
3. Add-ons, Secrets: **GITHUB_TOKEN**, switched on for this notebook.
4. **Save Version, Save & Run All (Commit)**. You can close the tab.

Output: `stage07_position_shuffle_outputs.zip`, also sent to the GitHub release `outputs-stage07_position_shuffle` after every step. If the session stops, run it again: finished steps are skipped.

In [ ]:
GITHUB_REPO = "FaiazPronoy/TriLens-AMR"
CODE_REF = "main"
NAME = "stage07_position_shuffle"
ATTENTION = "conditioned"
WRITES = ["position_shuffle_test"]
NEEDS = []
NEEDS_GPU = True
RESUME_OWN = True

import os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
BASE = Path("/tmp/trilens")
WORK = BASE / "work"
RES = WORK / "results"
OUT_ZIP = Path(f"/kaggle/working/{NAME}_outputs.zip")
PREFIX = "standard_attention/" if ATTENTION == "standard" else ""
STANDARD = ("stage05_standard_attention_random", "stage06_standard_attention_other_splits")
CONDITIONED = ("stage03_trilens_random", "stage04_trilens_other_splits")


def find_one(name):
    hits = sorted(INPUT.rglob(name))
    return hits[0] if hits else None


def secret(key):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key)
    except Exception:
        return os.environ.get(key)


TOKEN = secret("GITHUB_TOKEN")
print("github :", GITHUB_REPO, "(token found)" if TOKEN else "(no token: nothing is uploaded)")

repo = BASE / "repo"
if not repo.exists():
    auth = []
    if TOKEN:
        import base64
        auth = ["-c", "http.extraheader=AUTHORIZATION: basic "
                + base64.b64encode(f"x-access-token:{TOKEN}".encode()).decode()]
    r = subprocess.run(["git", *auth, "clone", "--quiet", f"https://github.com/{GITHUB_REPO}.git", str(repo)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git clone failed: " + (r.stderr or "").replace(TOKEN or "\0", "***"))
    subprocess.run(["git", "checkout", "--quiet", CODE_REF], cwd=repo, check=True)
CODE_VERSION = subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, capture_output=True, text=True).stdout.strip()
print("version:", CODE_VERSION)
if not WORK.exists():
    shutil.copytree(repo, WORK, ignore=lambda d, n: [x for x in n if (Path(d) == repo and x in
                    ("results", "data", "paper", "thesis", ".git")) or x == "__pycache__"])

master = find_one("arcadia_v2_master_features.pkl")
if master is None:
    raise FileNotFoundError("add the dataset trilens-data (Add Input)")
proc, giessen = WORK / "data" / "processed", WORK / "data" / "Giessen_dataset"
proc.mkdir(parents=True, exist_ok=True)
giessen.mkdir(parents=True, exist_ok=True)
for f in sorted(master.parent.iterdir()):
    if f.is_file():
        dst = (giessen if f.name.startswith("cip_ctx_ctz_gen") else proc) / f.name
        if not dst.exists():
            dst.symlink_to(f)
for rel in ("raw/strains_metadata.csv", "raw/pangenome/pangenome_cds.fa", "raw/pangenome/gene_presence_absence.csv"):
    f = find_one(Path(rel).name)
    dst = WORK / "data" / rel
    if f is not None and not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        dst.symlink_to(f)

gene_filtered = proc / "arcadia_v2_gene_filtered.pkl"
repaired = BASE / "repaired" / gene_filtered.name
if gene_filtered.is_symlink() and Path(os.readlink(gene_filtered)) != repaired:
    r = subprocess.run([sys.executable, "-m", "pipeline.step06_repair_snp_gene_blocks",
                        "--src", str(gene_filtered.resolve()),
                        "--snp", str((proc / "arcadia_v2_snp_matrix.pkl").resolve()),
                        "--out", str(repaired), "--report", str(RES / "diagnostics" / "data_repair.json")],
                       cwd=WORK, capture_output=True, text=True)
    if r.returncode == 0:
        gene_filtered.unlink()
        gene_filtered.symlink_to(repaired)
        print("data   : gene-filtered SNP blocks repaired")
    elif r.returncode != 2:
        raise RuntimeError("the SNP block repair refused the file:\n" + r.stdout[-2000:] + r.stderr[-2000:])

RES.mkdir(parents=True, exist_ok=True)
unfinished = set()


def target(stage, member):
    if ATTENTION == "standard":
        if stage in CONDITIONED:
            return None
        if member.startswith("standard_attention/"):
            member = member[len("standard_attention/"):]
    return RES / member


def take(stage, rel, read):
    t = target(stage, rel)
    if t is None or rel.split("/")[0].startswith("_") or (stage == NAME and t.exists()):
        return 0
    t.parent.mkdir(parents=True, exist_ok=True)
    t.write_bytes(read())
    return 1


def unpack(stage, zf):
    names = zf.namelist()
    if "_COMPLETE.txt" not in names:
        unfinished.add(stage)
    return sum(take(stage, m.filename, lambda m=m: zf.read(m)) for m in zf.infolist() if not m.is_dir())


have = set()
for z in sorted(INPUT.rglob("*_outputs.zip")):
    stage = z.name[: -len("_outputs.zip")]
    have.add(stage)
    with zipfile.ZipFile(z) as zf:
        print(f"earlier: {stage:42s} {unpack(stage, zf)} files")

sys.path.insert(0, str(WORK))
from trilens.utils import github_sync as gh
for stage in [s for s in ([NAME] if RESUME_OWN else []) + NEEDS if s not in have]:
    z = gh.download_asset(GITHUB_REPO, TOKEN, f"outputs-{stage}", f"{stage}_outputs.zip", BASE / "inputs")
    if z is None:
        if stage != NAME:
            print(f"earlier: {stage} NOT FOUND - run that notebook first or attach its output")
            unfinished.add(stage)
        continue
    with zipfile.ZipFile(z) as zf:
        print(f"earlier: {stage:42s} {unpack(stage, zf)} files (GitHub release)")

waiting = sorted(set(NEEDS) & unfinished)
if waiting:
    raise RuntimeError(f"not finished yet: {waiting}. Run this notebook after they say 'All steps finished'.")

if "stage01_gene_presence_matrix" in NEEDS:
    rebuilt = RES / "gene_presence_rebuild" / "arcadia_v2_gene_presence_absence.pkl"
    if not rebuilt.exists():
        raise RuntimeError("stage01_gene_presence_matrix finished without the rebuilt matrix")
    link = proc / rebuilt.name
    if link.exists() or link.is_symlink():
        link.unlink()
    link.symlink_to(rebuilt)
    print("data   : gene presence/absence -> rebuilt copy")

os.chdir(WORK)
os.environ.update(TRILENS_ROOT=str(WORK), TRILENS_ATTENTION=ATTENTION, WANDB_MODE="disabled",
                  PYTHONUNBUFFERED="1", MPLBACKEND="Agg", TRILENS_REPO_RESULTS=str(repo / "results"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
import torch
ENV_TEXT = "\n".join([f"python {sys.version.split()[0]}", f"torch {torch.__version__} cuda {torch.version.cuda}",
                      f"gpu {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}", "",
                      subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout])
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("model  :", "standard attention" if ATTENTION == "standard" else "strain-conditioned attention")
if NEEDS_GPU and not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4 x2, then Save & Run All again.")

In [ ]:
FAILED = []
last_sync = [0.0]


def keep(f):
    return not (set(f.parts) & {"cache", "ckpts", "__pycache__"} or f.suffix == ".ckpt")


def package(complete=False):
    tmp = OUT_ZIP.with_suffix(".tmp")
    dirs = [d for d in RES.iterdir() if d.is_dir()] if WRITES == ["*"] else [RES / d for d in WRITES]
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr(f"_STAGE_{NAME}.txt", NAME)
        z.writestr("_CODE_VERSION.txt", f"{NAME}\n{GITHUB_REPO}\n{CODE_VERSION}\n")
        z.writestr("_ENV.txt", ENV_TEXT)
        if complete:
            z.writestr("_COMPLETE.txt", "failed steps:\n" + "\n".join(FAILED))
        for d in dirs:
            for f in (d.rglob("*") if d.exists() else []):
                if f.is_file() and keep(f.relative_to(RES)):
                    z.write(f, PREFIX + f.relative_to(RES).as_posix())
    tmp.replace(OUT_ZIP)
    print(f"[saved] {OUT_ZIP.name}  {OUT_ZIP.stat().st_size / 1e6:.1f} MB")


def sync(force=False):
    if not TOKEN or (not force and time.time() - last_sync[0] < 600):
        return
    gh.upload_asset(GITHUB_REPO, TOKEN, f"outputs-{NAME}", OUT_ZIP)
    last_sync[0] = time.time()


def run(cmd, marker=None, cwd=None, env=None):
    root = Path(cwd) if cwd else WORK
    if marker and (root / "results" / marker).exists():
        print(f"[skip] {cmd}\n       (results/{marker} already exists)")
        return
    print(f"\n$ {cmd}\n", flush=True)
    t = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, env={**os.environ, **(env or {})})
    last_save = time.time()
    for line in p.stdout:
        print(line, end="")
        if time.time() - last_save > 1200:
            package()
            sync()
            last_save = time.time()
    p.wait()
    print(f"\n[{'ok' if p.returncode == 0 else 'FAILED'}] {cmd}  ({(time.time() - t) / 60:.1f} min)")
    package()
    sync()
    if p.returncode != 0:
        FAILED.append(cmd)

## giessen, orig

In [ ]:
run("SHUFFLE_ONLY=giessen_CIP_orig python -m pipeline.step17_test_position_shuffle --part run",
    "position_shuffle_test/runs/giessen_CIP_orig_run.json")

## giessen, shuf

In [ ]:
run("SHUFFLE_ONLY=giessen_CIP_shuf python -m pipeline.step17_test_position_shuffle --part run",
    "position_shuffle_test/runs/giessen_CIP_shuf_run.json")

## arcadia, orig

In [ ]:
run("SHUFFLE_ONLY=arcadia_CIP_orig python -m pipeline.step17_test_position_shuffle --part run",
    "position_shuffle_test/runs/arcadia_CIP_orig_run.json")

## arcadia, shuf

In [ ]:
run("SHUFFLE_ONLY=arcadia_CIP_shuf python -m pipeline.step17_test_position_shuffle --part run",
    "position_shuffle_test/runs/arcadia_CIP_shuf_run.json")

## All four runs together

In [ ]:
run("python -m pipeline.step17_test_position_shuffle --part run",
    "position_shuffle_test/runs/shuffle_results.csv")

## Summary

In [ ]:
run("python -m pipeline.step17_test_position_shuffle --part summary",
    "position_shuffle_test/verdict.json")

## Save

In [ ]:
package(complete=True)
sync(force=True)
print("Download stage07_position_shuffle_outputs.zip from the Output tab.")
if FAILED:
    print("\nThese steps FAILED (scroll up to their output):")
    for c in FAILED:
        print("  ", c)
    raise RuntimeError(f"{len(FAILED)} step(s) failed")
print("\nAll steps finished.")